In [10]:
import sys

import pandas as pd
import os
import glob

In [11]:
def extract_time_info(ts_int):
    total_hm = ts_int // 100
    
    hour = total_hm // 100
    minute = total_hm % 100
    
    total_minutes = hour * 60 + minute
    
    return hour, minute, total_minutes

In [12]:
def timetable_2_pieceofwork(file_path,split_stop):
    timetable_raw = pd.read_csv(file_path)
    timetable_raw['timestamp'] = pd.to_numeric(timetable_raw['timestamp'], errors='coerce')

    rows = []
    new_trip_id = 1
    for trip_index, group in timetable_raw.groupby('trip_index'):
        group = group.sort_values('timestamp')
        split_mask = group['stop_name'] == split_stop
        if split_mask.any():
            split_seq = group[split_mask].index[0]
            is_first = split_seq == group.index[0]
            is_last = split_seq == group.index[-1]
        if split_mask.any() and not is_first and not is_last:
            split_seq = group[split_mask].index[0]
            piece1 = group.loc[:split_seq].copy()
            piece1['trip_index'] = new_trip_id
            new_trip_id += 1
            # piece 2：从CS Centrumzijde（含）到终点
            piece2 = group.loc[split_seq:].copy()
            piece2['trip_index'] = new_trip_id
            new_trip_id += 1

            rows.append(piece1)
            rows.append(piece2)
        else:
            group = group.copy()
            group['trip_index'] = new_trip_id
            new_trip_id += 1
            rows.append(group)

    timetable_raw = pd.concat(rows, ignore_index=True)
    timetable_piece = timetable_raw.sort_values(['trip_index', 'timestamp']).groupby('trip_index').agg(
        start_stop=('stop_name', 'first'),
        end_stop=('stop_name', 'last'),
        start_time=('timestamp', 'first'),
        end_time=('timestamp', 'last')
    ).reset_index()

    timetable_piece[['start_hour', 'start_min', 'start_total_min']] = timetable_piece['start_time'].apply(
        lambda x: pd.Series(extract_time_info(x))
    )

    timetable_piece[['end_hour', 'end_min', 'end_total_min']] = timetable_piece['end_time'].apply(
        lambda x: pd.Series(extract_time_info(x))
    )

    timetable_piece['duration_min'] = timetable_piece['end_total_min'] - timetable_piece['start_total_min']

    timetable_output = timetable_piece.drop(columns=["start_time","end_time","start_total_min","end_total_min"])

    return timetable_output

In [15]:
def folder_runthrought(input_dir, output_dir,split_stop):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    file_pattern = os.path.join(input_dir, "uov_timetable_*.csv")
    input_files = glob.glob(file_pattern)
    
    for file_path in input_files:

        base_name = os.path.basename(file_path)
        
        output_name = base_name.replace("timetable", "pow")
        output_path = os.path.join(output_dir, output_name)
        
        print(f"processing: {base_name} -> {output_name}")
        
        # read file and call timetable_2_pieceofwork
        try:
            print(file_path)
            
            df_pow = timetable_2_pieceofwork(file_path,split_stop)
            
            df_pow.to_csv(output_path, index=False)
            
        except Exception as e:
            print(f"An error occurred while processing {base_name}: {e}")

In [16]:
if __name__ == "__main__":
    SPLIT_STOP = 'CS Centrumzijde'
    input_folder = 'timetable_bus1to8'
    output_folder = 'pow_bus1to8'
    folder_runthrought(input_folder, output_folder,SPLIT_STOP)

processing: uov_timetable_1.csv -> uov_pow_1.csv
timetable_bus1to8\uov_timetable_1.csv
processing: uov_timetable_1_rev.csv -> uov_pow_1_rev.csv
timetable_bus1to8\uov_timetable_1_rev.csv
processing: uov_timetable_2.csv -> uov_pow_2.csv
timetable_bus1to8\uov_timetable_2.csv
processing: uov_timetable_28.csv -> uov_pow_28.csv
timetable_bus1to8\uov_timetable_28.csv
processing: uov_timetable_28_rev.csv -> uov_pow_28_rev.csv
timetable_bus1to8\uov_timetable_28_rev.csv
processing: uov_timetable_2_rev.csv -> uov_pow_2_rev.csv
timetable_bus1to8\uov_timetable_2_rev.csv
processing: uov_timetable_3.csv -> uov_pow_3.csv
timetable_bus1to8\uov_timetable_3.csv
processing: uov_timetable_3_rev.csv -> uov_pow_3_rev.csv
timetable_bus1to8\uov_timetable_3_rev.csv
processing: uov_timetable_5.csv -> uov_pow_5.csv
timetable_bus1to8\uov_timetable_5.csv
processing: uov_timetable_5_rev.csv -> uov_pow_5_rev.csv
timetable_bus1to8\uov_timetable_5_rev.csv
processing: uov_timetable_6.csv -> uov_pow_6.csv
timetable_bus1t